# Guia de Engenharia de IA: Heurística de Pontuação & Monte Carlo Tree Search (MCTS) 🌌

Este notebook oferece uma explicação didática, prática e analítica sobre duas das principais arquiteturas de tomada de decisão em sistemas competitivos e jogos de simulação: **Sistemas de Pontuação Baseados em Heurísticas (Scoring Systems)** e **Busca em Árvore de Monte Carlo (MCTS)**.

Ele foi projetado como um material de referência e aprendizagem técnica, detalhando o funcionamento, as equações matemáticas, os cenários práticos de uso e as implementações específicas que aplicamos no projeto **Orbit Wars**.

## 1. O Paradigma de Tomada de Decisão em Ambientes Competitivos

Em Orbit Wars, os agentes competem em um ambiente dinâmico com regras rígidas de física orbital, atração gravitacional (Colisão Solar) e restrições geométricas de espaço. A tomada de decisão ocorre em tempo real de forma fechada (Closed-Loop), onde a cada passo de tempo (Tick) o agente recebe a observação completa do estado físico do jogo e deve decidir:

1. **De qual planeta aliado** lançar forças (`from_planet_id`).
2. **Qual a direção orbital** para mirar e disparar (`direction_angle`).
3. **Quantas naves** enviar no ataque/defesa (`num_ships`).

Para responder a essas três perguntas, diferentes paradigmas de controle podem ser empregados. A seguir, estudaremos as duas principais ferramentas de planejamento determinístico.

## 2. Heurística de Pontuação (Scoring System)

### O Que é?
Um **Sistema de Pontuação por Heurística** é uma abordagem determinística direta baseada em regras de negócio. Em vez de simular exaustivamente turnos futuros, a heurística foca em avaliar o estado atual instantâneo e atribuir uma nota matemática (Score) para cada par de ação-alvo possível. O agente simplesmente executa a ação que obteve a maior pontuação global.

### Formulação Matemática Geral
Dado um planeta de origem $P$ e um candidato a alvo $T$, a pontuação heurística é geralmente modelada como uma **soma ponderada (Weighted Sum)** de características táticas:

$$\text{Score}(P \to T) = w_{\text{prod}} \cdot f_{\text{prod}}(T) - w_{\text{dist}} \cdot f_{\text{dist}}(P, T) - w_{\text{cost}} \cdot f_{\text{cost}}(P, T) + \text{Bônus}_T$$

Onde:
* $f_{\text{prod}}(T)$ representa a capacidade produtiva futura que o planeta alvo $T$ nos dará após a captura.
* $f_{\text{dist}}(P, T)$ representa o atraso orbital/tempo de tráfego espacial (quanto mais longe, maior o risco de reação inimiga).
* $f_{\text{cost}}(P, T)$ representa o volume de naves que precisamos investir na conquista.
* $w_i$ são os pesos (Weights) que definem a agressividade, o foco econômico ou o conservadorismo do agente.

### Por Que Usamos no Nosso Projeto?
A heurística aprimorada (`heuristic_scorer.py`) é o motor líder do nosso projeto, alcançando **88.2% de taxa de vitória** em nossa arena local. Os principais motivos da escolha são:
1. **Tempo de Resposta Instantâneo**: Executa em menos de $1\text{ ms}$, muito abaixo do limite crítico de $100\text{ ms}$ imposto pelo Kaggle.
2. **Previsibilidade Tática**: O comportamento é totalmente determinístico. Se o agente comete um erro, conseguimos ajustar a fórmula ou os pesos diretamente com precisão cirúrgica.
3. **Física Preditiva Acoplada**: Conseguimos embutir regras de física geométrica de alta precisão (cálculo de colisão solar por intersecção vetorial, desvio planetário por Ray Casting e cálculo da posição orbital futura do alvo no turno exato de chegada).

### Inovações Implementadas no Nosso Agente:
* **Stateful Targeting Tracker (`self.sent_fleets_tracker`)**: Um dicionário em memória que impede ataques duplicados ou desperdício de forças para um planeta alvo que já possui uma frota aliada a caminho. Ele destrava o planeta de origem para atacar outros vizinhos de forma coordenada.
* **Defesa Proporcional Dinâmica**: Uma reserva de defesa que escala proporcionalmente (de 5 a 35 naves) baseada no número de bases que controlamos, impedindo paralisia na fase de expansão inicial e garantindo proteção máxima contra snipes nas fases avançadas.

In [ ]:
# Exemplo Prático de Avaliação por Pontuação Heurística
import math

class Planeta:
    def __init__(self, pid, x, y, production, ships, owner):
        self.id = pid
        self.x = x
        self.y = y
        self.production = production
        self.ships = ships
        self.owner = owner  # -1: neutro, 0: jogador, 1: inimigo

# Pesos calibrados da heurística
production_weight = 12.0
distance_weight = 3.5
ship_cost_weight = 0.1

def calcular_distancia(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def avaliar_alvos_heuristica(meu_planeta, alvos):
    print(f"=== Planeta Origem {meu_planeta.id} (Disponível: {meu_planeta.ships} naves) ===")
    melhor_alvo = None
    melhor_score = -float('inf')
    
    for alvo in alvos:
        dist = calcular_distancia(meu_planeta, alvo)
        # Naves propostas para conquista garantida (+2 de segurança)
        naves_propostas = alvo.ships + 2
        
        # Cálculo do Score
        score = (
            (production_weight * alvo.production) -
            (distance_weight * dist) -
            (ship_cost_weight * naves_propostas)
        )
        
        tipo_alvo = "Neutro" if alvo.owner == -1 else "Inimigo"
        print(f"-> Candidato {alvo.id} ({tipo_alvo}) | Prod: {alvo.production} | Dist: {dist:.1f} | Naves: {naves_propostas} | SCORE: {score:.2f}")
        
        if score > melhor_score:
            melhor_score = score
            melhor_alvo = alvo
            
    print(f"\nDecisão: Atacar Planeta {melhor_alvo.id} com Score {melhor_score:.2f}!\n")

# Criando instâncias de teste
meu_planeta = Planeta(pid=0, x=50.0, y=50.0, production=5, ships=50, owner=0)
neutro_proximo = Planeta(pid=1, x=60.0, y=50.0, production=3, ships=10, owner=-1)
inimigo_distante_rico = Planeta(pid=2, x=80.0, y=50.0, production=8, ships=15, owner=1)

avaliar_alvos_heuristica(meu_planeta, [neutro_proximo, inimigo_distante_rico])

## 3. Monte Carlo Tree Search (MCTS)

### O Que é?
O **Monte Carlo Tree Search (Busca em Árvore de Monte Carlo)** é um algoritmo de planejamento probabilístico tático de longo prazo. Em vez de tomar decisões reativas baseadas apenas no cenário imediato, o MCTS constrói de forma incremental uma árvore de decisões simulando milhares de rodadas futuras rápidas (Rollouts) até o fim da partida, determinando estatisticamente qual decisão atual maximiza a probabilidade real de vitória.

### As 4 Fases Clássicas do Loop MCTS:
A cada passo de decisão, o MCTS executa repetidamente as seguintes etapas:

1. **Seleção**: A partir do nó raiz (estado físico atual), navegamos pela árvore escolhendo os nós com base no balanço de exploração e explotação até alcançar um nó folha.
2. **Expansão**: Se o nó folha não representa um estado terminal (fim do jogo), criamos novos nós filhos correspondentes a ações ainda não exploradas.
3. **Simulação (Rollout)**: A partir do novo nó expansor, rodamos uma simulação de jogo completa de forma ultra-rápida (usando uma política barata, como ações aleatórias ou a própria heurística) para ver se o jogo resulta em Vitória ($1$) ou Derrota ($0$).
4. **Retropropagação (Backpropagation)**: Subimos de volta do nó simulado até a raiz, incrementando o contador de visitas ($N_i$) e atualizando a contagem de vitórias ($W_i$) de todos os nós ancestrais.

### A Equação de Seleção: UCT (Upper Confidence Bound for Trees)
Para guiar a fase de Seleção de forma ótima, garantindo que a busca não sofra de visão túnel (explotação prematura) nem de desperdício computacional (exploração excessiva), aplicamos a fórmula UCT:

$$UCT_i = \frac{W_i}{N_i} + c \sqrt{\frac{\ln N_p}{N_i}}$$

Onde:
* $W_i$: Número de vitórias acumuladas a partir do nó $i$.
* $N_i$: Visitas totais do nó $i$.
* $N_p$: Visitas totais do nó pai de $i$.
* $\frac{W_i}{N_i}$: **Explotação (Exploitation)** - O Win Rate empírico. Prioriza caminhos que historicamente deram certo.
* $c$: Constante de exploração (teoricamente $\sqrt{2} \approx 1.414$, ajustada empiricamente). Encoraja testar novos caminhos.
* $\sqrt{\frac{\ln N_p}{N_i}}$: **Exploração (Exploration)** - Dá bônus aos caminhos pouco explorados. Conforme o número de visitas do pai aumenta, a nota dos filhos pouco visitados decola.

### Por Que Usamos no Nosso Projeto?
O Orbit Wars é um jogo de simulação física complexo. O MCTS clássico exigiria simular cada mudança de coordenada física de dezenas de frotas e trajetórias a cada tick, o que é inviável dentro do limite de $100\text{ ms}$.

**Nossa Solução Híbrida (`mcts_search.py`)**:
Construímos um **MCTS Macro-Abstrato**. 
* Os nós da nossa árvore não representam movimentos milimétricos, mas sim ações estratégicas discretas (ex: "Atacar Alvo A", "Defender Base B").
* A fase de **Rollout** não simula a física passo a passo, mas sim avalia a mudança de posse de planetas usando a nossa heurística barata baseada em distâncias e taxas de produção.
* Isso nos dá o melhor dos dois mundos: o poder de **antecipar armadilhas táticas de múltiplos turnos** de forma incrivelmente leve!

In [ ]:
# Exemplo Prático de Seleção de Jogada MCTS usando a Fórmula UCT
import math

class NodoMCTS:
    def __init__(self, jogada, pai=None):
        self.jogada = jogada
        self.pai = pai
        self.filhos = []
        self.visitas = 0
        self.vitorias = 0.0

    def calcular_uct(self, c=1.414):
        if self.visitas == 0:
            # Força a exploração imediata se o nó nunca foi testado
            return float('inf')
        
        # Explotação: Win Rate empírico
        explotacao = self.vitorias / self.visitas
        # Exploração: Bônus por falta de informação
        exploracao = c * math.sqrt(math.log(self.pai.visitas) / self.visitas)
        
        return explotacao + exploracao

def rodar_selecao_mcts(raiz):
    print(f"=== Selecionando Jogada MCTS (Visitas do Pai: {raiz.visitas}) ===")
    for filho in raiz.filhos:
        uct = filho.calcular_uct()
        win_rate = (filho.vitorias / filho.visitas * 100) if filho.visitas > 0 else 0
        print(f"Nodo '{filho.jogada}' | Visitas: {filho.visitas} | Win Rate: {win_rate:.1f}% | UCT: {uct:.4f}")
        
    escolhido = max(raiz.filhos, key=lambda n: n.calcular_uct())
    print(f"\nDecisão UCT: Expandir/Simular a jogada '{escolhido.jogada}'!\n")
    return escolhido

# Criando a raiz
raiz_estado_atual = NodoMCTS("Turno 45")
raiz_estado_atual.visitas = 100

# Nós filhos correspondentes a decisões macro possíveis
opcao_agressiva = NodoMCTS("Contra-ataque Massivo no Inimigo", pai=raiz_estado_atual)
opcao_agressiva.visitas = 60
opcao_agressiva.vitorias = 38.0  # ~63% win rate

opcao_segura = NodoMCTS("Expandir para Neutro Adjacente", pai=raiz_estado_atual)
opcao_segura.visitas = 30
opcao_segura.vitorias = 24.0  # 80% win rate

opcao_defensiva = NodoMCTS("Dobrar Guardas nas Fronteiras", pai=raiz_estado_atual)
opcao_defensiva.visitas = 10
opcao_defensiva.vitorias = 2.0   # 20% win rate (muito pouco visitado)

raiz_estado_atual.filhos = [opcao_agressiva, opcao_segura, opcao_defensiva]

rodar_selecao_mcts(raiz_estado_atual)

## 4. Matriz Comparativa: Heurística vs. MCTS vs. PPO (Reinforcement Learning)

Esta matriz ajuda a equipe a tomar decisões arquiteturais informadas, avaliando as três abordagens estudadas:

| Dimensão | Heurística de Pontuação | Busca em Árvore (MCTS) | Aprendizado por Reforço (PPO) |
| :--- | :--- | :--- | :--- |
| **Tempo de Execução** | Instantâneo ($< 1\text{ ms}$) | Variável ($10 - 80\text{ ms}$) | Rápido (passagem de rede, $\sim 2\text{ ms}$) |
| **Foresight (Passos Futuros)** | Baixo (mira imediata) | Excelente (simula ramificações) | Médio (codificado na recompensa acumulada) |
| **Complexidade de Engenharia** | Baixa (fórmulas diretas) | Alta (requer simulador interno) | Altíssima (wrapper Gym, extração de features) |
| **Custo de Dados (Treino)** | Zero | Zero (busca puramente local) | Altíssimo (milhões de steps de treino) |
| **Segurança e Controle** | $100\%$ controlável | Segura e robusta | Sujeita a ações caóticas fora de distribuição |

## 5. Conclusão e Próximos Passos

1. **Heurísticas de Pontuação** são ferramentas fantásticas de controle analítico. O nosso *Improved Heuristic* atingiu vitória em $88.2\%$ das seeds da arena local devido à calibração física precisa e desvios de obstáculos de alta fidelidade.
2. **Busca probabilística (MCTS)** é ideal para resolver cenários onde a ação do oponente reage taticamente à nossa de forma complexa, mas esbarra na barreira do custo computacional do simulador físico.

### Referências Acadêmicas Recomendadas:
* Browne, C. B. et al. (2012). *A Survey of Monte Carlo Tree Search Methods*. IEEE Transactions on Computational Intelligence.
* Silver, D. et al. (2016). *Mastering the game of Go with deep neural networks and tree search*. Nature.
* Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction*. MIT Press.

---
*Este material é parte dos artefatos oficiais de governança e educação técnica do projeto Orbit Wars.*